# Country Missingness Scoring

Identifies countries systematically absent from data they should have.
Classifies each (country, year) as dissolved, microstate, failed, degraded, reporting, or strong.

**Phase 1: Model Definition**

In [1]:
using Revise
using InteractiveUtils

includet("phase1/functions/load_phase1.jl")

╔══════════════════════════════════════════════════════════════════════════╗
║ QoG METADATA JOINING - PHASE 0 LOADED                                ║
╚══════════════════════════════════════════════════════════════════════════╝

Quick Start:
    metadata = join_metadata()              # Run full pipeline (single isomorphism check)
    metadata = join_metadata_with_cascade()  # Run cascade (strictest → loosest), then union on slug
    quick_check()                             # Diagnostic check
    inspect_exceptions()                      # Review configuration
    show_usage()                              # Detailed documentation

Pipeline Steps:
    1. ingest_and_normalize()            # Load & normalize sources (PDF = qog_slugs_temporal.csv; min_year/max_year ingested)
    2. align_id_variables!(...)          # Harmonize ID vars
    3. run_isomorphism_cascade(...)      # Strictest → loosest until success; returns (stata_df, pdf_df, arrow_df) for union on slug
    4. unify_and_join(..

In [2]:
using CSV, DataFrames

df = load_augmented_or_build()
meta_df = CSV.read("data/qog_metadata_plus2.csv", DataFrame)
println("Loaded: $(nrow(df)) rows, $(nrow(meta_df)) slugs")

✓ Checksum verified: data/qog_std_ts_jan25_aug.arrow
✓ Loaded: 12391 rows × 2014 cols from data/qog_std_ts_jan25_aug.arrow
  ggis_rowid unique: ✓ | Required columns: ✓ | Missing regions: 0 ✓
Loaded: 12391 rows, 2010 slugs


## Run Pipeline

In [3]:
result = run_country_missingness(df, meta_df)

Step 0 — Country Temporal Profiles
    Total countries: 200
    Active: 194
    Dissolved: 6
      CSK Czechoslovakia (dissolved 1992, last data 1992)
      DDR German Democratic Republic (dissolved 1990, last data 1990)
      YMD Yemen Democratic (dissolved 1990, last data 1989)
      SUN USSR (dissolved 1991, last data 1991)
      YUG Yugoslavia (dissolved 1992, last data 1991)
      XTI Tibet (dissolved 1959, last data 1950)
    Microstates: 11
      AND Andorra (pop 0K)
      ATG Antigua and Barbuda (pop 0K)
      DMA Dominica (pop 0K)
      LIE Liechtenstein (pop 0K)
      MCO Monaco (pop 0K)
      NRU Nauru (pop 0K)
      MHL Marshall Islands (pop 0K)
      PLW Palau (pop 0K)
      KNA Saint Kitts and Nevis (pop 0K)
      SMR San Marino (pop 0K)
      TUV Tuvalu (pop 0K)

Step 1 — Per-Country, Per-Year Global Slug Coverage
    Global slugs in data: 616
    Computing coverage for each (country, year)...
    Scored 12369 (country, year) pairs
    Lag-affected years: 776

Step 2 — C

(profiles = 200×13 DataFrame
 Row │ ident_ccode  ident_ccodealp  ident_cname                        country ⋯
     │ Int64        String          String                             Int64   ⋯
─────┼──────────────────────────────────────────────────────────────────────────
   1 │           4  AFG             Afghanistan                                ⋯
   2 │           8  ALB             Albania
   3 │          12  DZA             Algeria
   4 │          20  AND             Andorra
   5 │          24  AGO             Angola                                     ⋯
   6 │          28  ATG             Antigua and Barbuda
   7 │          31  AZE             Azerbaijan
   8 │          32  ARG             Argentina
   9 │          36  AUS             Australia                                  ⋯
  10 │          40  AUT             Austria
  11 │          44  BHS             Bahamas (the)
  ⋮  │      ⋮             ⋮                         ⋮                          ⋱
 191 │         840  USA      

## Country Profiles

Failed, degraded, dissolved, and micro states, and temporal spans.

In [6]:
# Failed countries
failed = filter(r -> r.country_status == "failed", result.status)
failed_countries = unique(failed.ident_ccode)
for ccode in failed_countries
    rows = filter(r -> r.ident_ccode == ccode, result.status)
    prof = filter(r -> r.ident_ccode == ccode, result.profiles)
    alpha = prof.ident_ccodealp[1]
    name = prof.ident_cname[1]
    n_failed = count(==("failed"), rows.country_status)
    years = sort(filter(r -> r.country_status == "failed", rows).ident_year)
    println("  $alpha $(rpad(name, 35)) $n_failed failed years: $(first(years))–$(last(years))")
end


  TWN Taiwan (Province of China)          8 failed years: 2015–2022
  COD Congo (the Democratic Republic of the) 1 failed years: 1960–1960
  CSK Czechoslovakia                      2 failed years: 1991–1992
  DDR German Democratic Republic          1 failed years: 1990–1990
  MYS Malaysia                            3 failed years: 1960–1962
  VCT Saint Vincent and the Grenadines    1 failed years: 1980–1980
  SRB Serbia                              14 failed years: 1992–2005
  VNM Viet Nam                            14 failed years: 1963–1976
  ZWE Zimbabwe                            6 failed years: 1960–1965
  YMD Yemen Democratic                    18 failed years: 1971–1989
  SDN Sudan (the)                         12 failed years: 2012–2023
  SUN USSR                                22 failed years: 1970–1991
  YUG Yugoslavia                          1 failed years: 1991–1991


In [7]:
# Degraded countries
degraded = filter(r -> r.country_status == "degraded", result.status)
degraded_countries = unique(degraded.ident_ccode)
for ccode in degraded_countries
    rows = filter(r -> r.ident_ccode == ccode, result.status)
    prof = filter(r -> r.ident_ccode == ccode, result.profiles)
    alpha = prof.ident_ccodealp[1]
    name = prof.ident_cname[1]
    n_deg = count(==("degraded"), rows.country_status)
    years = sort(filter(r -> r.country_status == "degraded", rows).ident_year)
    println("  $alpha $(rpad(name, 35)) $n_deg degraded years: $(first(years))–$(last(years))")
end


  AFG Afghanistan                         1 degraded years: 2024–2024
  ALB Albania                             2 degraded years: 2023–2024
  DZA Algeria                             2 degraded years: 2023–2024
  AGO Angola                              2 degraded years: 2023–2024
  AZE Azerbaijan                          2 degraded years: 2023–2024
  ARG Argentina                           2 degraded years: 2023–2024
  AUS Australia                           2 degraded years: 2023–2024
  AUT Austria                             2 degraded years: 2023–2024
  BHS Bahamas (the)                       1 degraded years: 2024–2024
  BHR Bahrain                             2 degraded years: 2023–2024
  BGD Bangladesh                          2 degraded years: 2023–2024
  ARM Armenia                             2 degraded years: 2023–2024
  BRB Barbados                            1 degraded years: 2024–2024
  BEL Belgium                             2 degraded years: 2023–2024
  BTN Bhutan        

In [4]:
# Dissolved states
filter(r -> r.is_dissolved, result.profiles)

Row,ident_ccode,ident_ccodealp,ident_cname,country_birth_year,country_death_year,n_years,year_span,is_active,is_dissolved,dissolution_year,is_microstate,max_pop,un_subregion_code
,Int64,String,String,Int64,Int64,Int64,Int64,Bool,Bool,Int64?,Bool,Float64?,Int64
1,200,CSK,Czechoslovakia,1946,1992,47,47,false,true,1992,false,missing,151
2,278,DDR,German Democratic Republic,1949,1990,42,42,false,true,1990,false,missing,155
3,720,YMD,Yemen Democratic,1968,1989,22,22,false,true,1990,false,missing,145
4,810,SUN,USSR,1946,1991,46,46,false,true,1991,false,missing,151
5,891,YUG,Yugoslavia,1946,1991,46,46,false,true,1992,false,missing,39
6,9156,XTI,Tibet,1946,1950,5,5,false,true,1959,false,missing,145


In [5]:
# Microstates
filter(r -> r.is_microstate, result.profiles)

Row,ident_ccode,ident_ccodealp,ident_cname,country_birth_year,country_death_year,n_years,year_span,is_active,is_dissolved,dissolution_year,is_microstate,max_pop,un_subregion_code
,Int64,String,String,Int64,Int64,Int64,Int64,Bool,Bool,Int64?,Bool,Float64?,Int64
1,20,AND,Andorra,1946,2024,79,79,true,false,missing,true,84.158,39
2,28,ATG,Antigua and Barbuda,1982,2024,43,43,true,false,missing,true,93.082,419
3,212,DMA,Dominica,1979,2024,46,46,true,false,missing,true,73.112,419
4,438,LIE,Liechtenstein,1946,2024,79,79,true,false,missing,true,39.46,155
5,492,MCO,Monaco,1946,2024,79,79,true,false,missing,true,39.136,155
6,520,NRU,Nauru,1968,2024,57,57,true,false,missing,true,11.845,57
7,584,MHL,Marshall Islands,1987,2024,38,38,true,false,missing,true,52.07,57
8,585,PLW,Palau,1995,2024,30,30,true,false,missing,true,19.902,57
9,659,KNA,Saint Kitts and Nevis,1984,2024,41,41,true,false,missing,true,47.093,419


## Status Distribution

How many country-years fall in each category?

In [ ]:
sort(combine(groupby(result.status, :country_status), nrow => :count), :count, rev=true)

## Failed & Degraded Countries

Which active countries are losing data coverage?

In [ ]:
# Countries ever classified as failed (excluding dissolved/micro)
failed = filter(r -> r.country_status == "failed", result.status)
failed_countries = unique(failed.ident_ccode)
println("Countries with 'failed' years: $(length(failed_countries))")

# Show their trajectory: status by decade
for ccode in failed_countries[1:min(10, length(failed_countries))]
    rows = filter(r -> r.ident_ccode == ccode, result.status)
    alpha = filter(r -> r.ident_ccode == ccode, result.profiles).ident_ccodealp[1]
    name = filter(r -> r.ident_ccode == ccode, result.profiles).ident_cname[1]
    println("\n  $alpha $name:")
    for decade_start in [1990, 2000, 2010, 2020]
        decade = filter(r -> decade_start <= r.ident_year < decade_start + 10, rows)
        if nrow(decade) > 0
            statuses = unique(decade.country_status)
            avg_cov = round(mean(decade.global_coverage_pct) * 100, digits=1)
            println("    $(decade_start)s: $(join(statuses, "/")) ($(avg_cov)% avg coverage)")
        end
    end
end

## Revised Slug Penetration

Slugs that gain penetration when failed states are excluded from the denominator.

In [ ]:
# Top gainers
first(result.penetration, 20)

In [ ]:
# How many slugs cross the 95% threshold with revised denominator?
original_global = count(r -> r.original_penetration >= 0.95, eachrow(result.penetration))
revised_global = count(r -> r.revised_penetration >= 0.95, eachrow(result.penetration))
println("Slugs ≥95% penetration:")
println("  Original denominator: $original_global")
println("  Revised denominator:  $revised_global")
println("  New globals:          $(revised_global - original_global)")

## Save Flags

In [ ]:
# CSV.write("data/country_missingness_flags.csv", result.flags)
# println("\u2705 Saved country_missingness_flags.csv")